# DS52 — Fondamentaux de la Science des Données
## Notebook d'accompagnement — Du débutant à l'expert

> **EDA · Preprocessing · Classification · Régression · Deep Learning · Big Data · XAI**  
> UTBM — Session 2026

Ce notebook suit **exactement** et **dans l'ordre** le kit technique DS52.  
Il vous permet d'importer un dataset depuis différentes sources et de dérouler tout le pipeline.

---


## ⚙️ Installation des dépendances

In [ ]:
# Installer les bibliothèques nécessaires (décommenter si besoin)
# !pip install pandas numpy scikit-learn matplotlib seaborn scipy xgboost lightgbm catboost
# !pip install imbalanced-learn shap lime optuna statsmodels pmdarima prophet
# !pip install torch torchvision transformers gensim dask[dataframe] pyspark
# !pip install umap-learn grad-cam

import warnings
warnings.filterwarnings('ignore')
print("✓ Prêt")


## 📦 Imports globaux

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RandomizedSearchCV, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder, PolynomialFeatures
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, log_loss, mean_squared_error, mean_absolute_error,
    r2_score, silhouette_score, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.ensemble import (
    RandomForestClassifier, IsolationForest, RandomForestRegressor
)
from sklearn.linear_model import (
    LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV, SGDClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, LocalOutlierFactor
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.feature_extraction.text import TfidfVectorizer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 30)
print("✓ Imports OK")


---
## 0. Processus Complet du Data Scientist

### 0.1 Chargement du dataset

Choisissez votre source de données parmi les options ci-dessous.


In [ ]:
# ══════════════════════════════════════════════════════════════
# CHARGEMENT DU DATASET — Choisir UNE source et l'activer
# ══════════════════════════════════════════════════════════════

SOURCE = "csv_local"  # Options : "csv_local", "csv_url", "sklearn_dataset",
                      #           "excel", "json", "parquet", "sql", "bank_demo"

# ──────────────────────────────────────────────────────────────
# Option 1 : CSV local
if SOURCE == "csv_local":
    PATH = "data.csv"           # ← remplacer par votre chemin
    SEPARATOR = ","             # "," ou ";"
    ENCODING = "utf-8"          # "utf-8" ou "latin-1"
    df_raw = pd.read_csv(PATH, sep=SEPARATOR, encoding=ENCODING)

# ──────────────────────────────────────────────────────────────
# Option 2 : CSV depuis une URL
elif SOURCE == "csv_url":
    URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df_raw = pd.read_csv(URL)

# ──────────────────────────────────────────────────────────────
# Option 3 : Dataset scikit-learn intégré
elif SOURCE == "sklearn_dataset":
    from sklearn.datasets import load_breast_cancer   # load_iris, load_wine, etc.
    data = load_breast_cancer()
    df_raw = pd.DataFrame(data.data, columns=data.feature_names)
    df_raw['target'] = data.target

# ──────────────────────────────────────────────────────────────
# Option 4 : Fichier Excel
elif SOURCE == "excel":
    PATH = "data.xlsx"          # ← remplacer par votre chemin
    SHEET = 0                   # nom ou index de l'onglet
    df_raw = pd.read_excel(PATH, sheet_name=SHEET)

# ──────────────────────────────────────────────────────────────
# Option 5 : JSON
elif SOURCE == "json":
    PATH = "data.json"          # ← remplacer par votre chemin
    df_raw = pd.read_json(PATH)

# ──────────────────────────────────────────────────────────────
# Option 6 : Parquet (format colonnaire compressé)
elif SOURCE == "parquet":
    PATH = "data.parquet"       # ← remplacer par votre chemin
    df_raw = pd.read_parquet(PATH)

# ──────────────────────────────────────────────────────────────
# Option 7 : Base de données SQL
elif SOURCE == "sql":
    import sqlite3              # ou sqlalchemy pour d'autres bases
    CONN_STR = "database.db"    # ← chemin ou chaîne de connexion
    QUERY = "SELECT * FROM ma_table"
    conn = sqlite3.connect(CONN_STR)
    df_raw = pd.read_sql(QUERY, conn)
    conn.close()

# ──────────────────────────────────────────────────────────────
# Option 8 : Bank Marketing Dataset (démo)
elif SOURCE == "bank_demo":
    URL = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/bank.csv"
    df_raw = pd.read_csv(URL, sep=";")

# ──────────────────────────────────────────────────────────────
print(f"✓ Dataset chargé : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
df_raw.head(3)


In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION : adapter à votre dataset
# ══════════════════════════════════════════════════════════════

TARGET_COL = "target"    # ← nom de la colonne cible
TASK = "classification"  # "classification" ou "regression"

# Vérifier que la cible existe
assert TARGET_COL in df_raw.columns, f"Colonne '{TARGET_COL}' introuvable. Colonnes : {list(df_raw.columns)}"
print(f"✓ Cible : '{TARGET_COL}' | Tâche : {TASK}")
print(f"Valeurs uniques de la cible : {df_raw[TARGET_COL].unique()[:10]}")


### 0.2 Pipeline du projet ML

| Étape | Action | Point clé |
|-------|--------|-----------|
| 1 | Compréhension métier | Définir l'objectif précis |
| 2 | Collecte des données | CSV, base de données, API, scraping |
| 3 | EDA | Explorer, visualiser, comprendre |
| 4 | Preprocessing | Nettoyer, encoder, normaliser |
| 5 | Modélisation | Choisir ≥ 3 algorithmes |
| 6 | Validation | Évaluer honnêtement |
| 7 | Optimisation | Hyperparamètres |
| 8 | Interprétation | Feature importance, SHAP |
| 9 | Big Data | Spark, Dask |
| 10 | Déploiement | API Flask/FastAPI |

> ⚠️ **Règle d'or** : ne jamais utiliser les données de test pour calibrer quoi que ce soit.


---
## 1. Types de Données & Encodage

### 1.1 Taxonomie automatique des variables


In [ ]:
# ── Détection automatique des types de variables ──────────────────
df = df_raw.copy()

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_COL in num_cols and TASK == "classification":
    num_cols.remove(TARGET_COL)

cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if TARGET_COL in cat_cols:
    cat_cols.remove(TARGET_COL)

print("Variables NUMÉRIQUES :", num_cols)
print("Variables CATÉGORIELLES :", cat_cols)
print(f"Variable CIBLE : {TARGET_COL}")
print(f"\nCardinalité des variables catégorielles :")
for c in cat_cols:
    print(f"  {c:30s} → {df[c].nunique()} valeurs uniques")


### 1.2 Encodage — Choisir la méthode

In [ ]:
# ── One-Hot Encoding (variables nominales) ────────────────────────
# Transforme une variable à k catégories en k colonnes binaires

# Exemple sur la 1ère variable catégorielle (si disponible)
if cat_cols:
    col_demo = cat_cols[0]
    print(f"=== One-Hot Encoding sur '{col_demo}' ===")
    demo_ohe = pd.get_dummies(df[[col_demo]], prefix=col_demo, drop_first=True)
    print(demo_ohe.head(5))
    print()

    # Avec sklearn
    from sklearn.preprocessing import OneHotEncoder
    ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    # ohe.fit_transform(df[[col_demo]])  # décommenter pour utiliser


In [ ]:
# ── Ordinal Encoding (variables ordinales) ─────────────────────────
# À adapter : définir l'ordre explicite de vos variables ordinales

ORDINAL_COLS = {}   # Exemple : {'education': ['primary','secondary','tertiary']}
# ORDINAL_COLS = {
#     'education': ['primary', 'secondary', 'tertiary'],
#     'rating': ['low', 'medium', 'high'],
# }

if ORDINAL_COLS:
    from sklearn.preprocessing import OrdinalEncoder
    for col, order in ORDINAL_COLS.items():
        enc = OrdinalEncoder(categories=[order])
        df[col] = enc.fit_transform(df[[col]])
        print(f"✓ OrdinalEncoder appliqué sur '{col}' : {order}")
else:
    print("ℹ Aucune variable ordinale définie (ORDINAL_COLS vide).")


In [ ]:
# ── Récapitulatif des méthodes d'encodage disponibles ──────────────
print("""
MÉTHODE          | USAGE                              | ATTENTION
-----------------|------------------------------------|--------------------------
One-Hot          | Variables nominales (peu de val.)  | Explosion dimensionnelle
Ordinal          | Variables ordinales (ordre naturel)| Ordre explicite requis
Label Encoding   | UNIQUEMENT pour les arbres         | Faux ordre pour LR/SVM
Target Encoding  | Haute cardinalité                  | Risque de leakage
Binary Encoding  | Haute cardinalité > 50 cat.        | Moins interprétable
Frequency Enc.   | Replace par la fréquence           | Perte d'information possible
Embedding (DL)   | Représentation dense apprise        | Nécessite beaucoup de données
""")


---
## 2. EDA — Analyse Exploratoire des Données

### 2.1 Checklist EDA complète


In [ ]:
# ── Aperçu global ────────────────────────────────────────────────
print("=== SHAPE ===")
print(df.shape)
print()
print("=== DTYPES ===")
print(df.dtypes)
print()
print("=== VALEURS MANQUANTES ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "Aucune valeur manquante")
print()
print("=== DOUBLONS ===")
print(f"Lignes dupliquées : {df.duplicated().sum()}")


In [ ]:
# ── Statistiques descriptives ────────────────────────────────────
df.describe(include='all').T.style.background_gradient(cmap='Blues', axis=0)


In [ ]:
# ── Distribution de la cible ─────────────────────────────────────
print("=== DISTRIBUTION DE LA CIBLE ===")
print(df[TARGET_COL].value_counts(normalize=True).round(4) * 100)
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET_COL].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f"Distribution de '{TARGET_COL}'", fontweight='bold')
axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=45)

df[TARGET_COL].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=sns.color_palette('muted'))
axes[1].set_ylabel("")
axes[1].set_title("Proportion", fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Distributions des variables numériques ───────────────────────
if num_cols:
    n = len(num_cols)
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = axes.flatten() if n > 1 else [axes]
    for i, col in enumerate(num_cols):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='steelblue')
        axes[i].set_title(col, fontweight='bold')
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Distributions des variables numériques", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Boxplots (outliers) ──────────────────────────────────────────
if num_cols:
    n = len(num_cols)
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = axes.flatten() if n > 1 else [axes]
    for i, col in enumerate(num_cols):
        sns.boxplot(y=df[col].dropna(), ax=axes[i], color='lightcoral')
        axes[i].set_title(col, fontweight='bold')
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Boxplots — détection visuelle des outliers", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Matrice de corrélation ───────────────────────────────────────
if len(num_cols) > 1:
    fig, ax = plt.subplots(figsize=(min(14, len(num_cols)+2), min(12, len(num_cols)+2)))
    corr_matrix = df[num_cols].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=len(num_cols) <= 15,
                fmt='.2f', cmap='RdBu_r', center=0,
                linewidths=0.5, ax=ax, vmin=-1, vmax=1)
    ax.set_title("Matrice de corrélation (triangle inférieur)", fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Corrélations fortes (> 0.7)
    strong = [(c1, c2, corr_matrix.loc[c1, c2])
              for i, c1 in enumerate(num_cols)
              for j, c2 in enumerate(num_cols)
              if i < j and abs(corr_matrix.loc[c1, c2]) > 0.7]
    if strong:
        print("⚠ Corrélations fortes (|r| > 0.7) — risque de multicolinéarité :")
        for c1, c2, r in strong:
            print(f"  {c1} ↔ {c2} : r = {r:.3f}")
    else:
        print("✓ Pas de corrélation forte détectée (|r| < 0.7)")


In [ ]:
# ── Valeurs moyennes par classe ───────────────────────────────────
if num_cols and TASK == "classification":
    print("=== MOYENNES PAR CLASSE (signaux discriminants) ===")
    print(df.groupby(TARGET_COL)[num_cols].mean().T.round(3))


### 2.2 Identifier les outliers

In [ ]:
# ── Méthode IQR (recommandée) ────────────────────────────────────
print("=== OUTLIERS — Méthode IQR ===")
outlier_report = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR
    n_out = ((df[col] < borne_inf) | (df[col] > borne_sup)).sum()
    pct = n_out / len(df) * 100
    outlier_report[col] = {'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                            'borne_inf': borne_inf, 'borne_sup': borne_sup,
                            'n_outliers': n_out, 'pct_outliers': pct}
    if n_out > 0:
        print(f"  {col:30s} → {n_out:5d} outliers ({pct:.1f}%) "
              f"| bornes [{borne_inf:.2f}, {borne_sup:.2f}]")

if not any(v['n_outliers'] > 0 for v in outlier_report.values()):
    print("  Aucun outlier détecté avec la méthode IQR.")


In [ ]:
# ── Méthode Z-score ──────────────────────────────────────────────
# Suppose une distribution normale — moins robuste que IQR
if num_cols:
    z_scores = np.abs(stats.zscore(df[num_cols].fillna(df[num_cols].median())))
    n_zscore_out = (z_scores > 3).any(axis=1).sum()
    print(f"Z-score (|z|>3) : {n_zscore_out} lignes ({n_zscore_out/len(df)*100:.1f}%)")


In [ ]:
# ── Isolation Forest (approche ML) ──────────────────────────────
CONTAMINATION = 0.05   # proportion estimée d'outliers

if num_cols:
    iso = IsolationForest(contamination=CONTAMINATION, random_state=42)
    X_num = df[num_cols].fillna(df[num_cols].median())
    iso_labels = iso.fit_predict(X_num)  # -1 = outlier, 1 = normal
    n_iso = (iso_labels == -1).sum()
    print(f"Isolation Forest : {n_iso} outliers détectés ({n_iso/len(df)*100:.1f}%)")
    
    # Choisir la méthode de traitement
    OUTLIER_TREATMENT = "keep"  # "keep", "remove_iqr", "remove_iso", "cap_iqr"
    
    if OUTLIER_TREATMENT == "remove_iqr":
        mask = pd.Series([True] * len(df), index=df.index)
        for col in num_cols:
            r = outlier_report[col]
            mask &= (df[col] >= r['borne_inf']) & (df[col] <= r['borne_sup'])
        df = df[mask]
        print(f"✓ Outliers IQR supprimés : {len(df)} lignes restantes")
        
    elif OUTLIER_TREATMENT == "remove_iso":
        df = df[iso_labels == 1]
        print(f"✓ Outliers Isolation Forest supprimés : {len(df)} lignes restantes")
        
    elif OUTLIER_TREATMENT == "cap_iqr":
        for col in num_cols:
            r = outlier_report[col]
            df[col] = df[col].clip(lower=r['borne_inf'], upper=r['borne_sup'])
        print("✓ Outliers plafonnés (winsorisation IQR)")
        
    else:
        print("ℹ Outliers conservés (OUTLIER_TREATMENT = 'keep')")


### 2.3 Traitement des valeurs manquantes

In [ ]:
# ── Bilan des valeurs manquantes ──────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'n_missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['n_missing'] > 0].sort_values('pct', ascending=False)

if len(missing_df) > 0:
    print("=== VALEURS MANQUANTES ===")
    print(missing_df)
    print()
    
    # Visualisation
    fig, ax = plt.subplots(figsize=(10, max(3, len(missing_df) * 0.5)))
    missing_df['pct'].plot(kind='barh', ax=ax, color='coral', edgecolor='white')
    ax.axvline(x=5, color='orange', linestyle='--', label='5%')
    ax.axvline(x=30, color='red', linestyle='--', label='30%')
    ax.set_xlabel("% de valeurs manquantes")
    ax.set_title("Valeurs manquantes par colonne", fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("✓ Aucune valeur manquante")


In [ ]:
# ── Choix de la stratégie d'imputation ───────────────────────────
# Les imputeurs sont intégrés dans le Pipeline (section 4) — pas de leakage possible

IMPUTER_NUM = "median"    # "median", "mean", "knn", "iterative"
IMPUTER_CAT = "most_frequent"  # "most_frequent", "constant" (ajoute 'unknown')

print(f"Stratégie numérique  : {IMPUTER_NUM}")
print(f"Stratégie catégorielle : {IMPUTER_CAT}")
print()
print("""⚠ Les imputeurs sont appliqués via Pipeline scikit-learn (section 4).
  → fit() sur X_train uniquement → pas de data leakage.""")


### 2.4 Déséquilibre des classes

In [ ]:
# ── Analyse du déséquilibre ──────────────────────────────────────
if TASK == "classification":
    counts = df[TARGET_COL].value_counts()
    ratio_max = counts.max() / counts.min() if len(counts) > 1 else 1
    print(f"Distribution des classes :")
    print(counts)
    print()
    print(f"Ratio majority/minority : {ratio_max:.1f}x")
    
    if ratio_max > 1.5:
        print(f"⚠ Déséquilibre détecté (ratio > 1.5). Stratégies recommandées :")
        print("   1. class_weight='balanced' dans le modèle")
        print("   2. SMOTE (suréchantillonnage synthétique)")
        print("   3. RandomUnderSampler")
        print("   4. Ajustement du seuil de décision")
        print("   5. Métrique : AUC-ROC ou F1, PAS accuracy")
    else:
        print("✓ Classes équilibrées")
    
    # Choix de la stratégie (activée dans le pipeline section 5)
    IMBALANCE_STRATEGY = "class_weight"  # "class_weight", "smote", "none"
    print(f"\nStratégie retenue : {IMBALANCE_STRATEGY}")


---
## 3. Apprentissage Statistique — Fondements

### 3.1 Biais, Variance et compromis fondamental


In [ ]:
# ── Illustration Biais-Variance ──────────────────────────────────
print("""
SOURCE     | CAUSE                       | SYMPTÔME               | REMÈDE
-----------|-----------------------------|-----------------------|-----------------
Biais      | Modèle trop simple          | Underfitting           | Modèle + complexe
Variance   | Modèle trop complexe        | Overfitting            | Régularisation
Bruit      | Bruit inhérent aux données  | Erreur plancher        | Aucun
""")

# Visualisation de la courbe d'apprentissage (Learning Curve)
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, X, y, cv=5, scoring='roc_auc' if TASK=='classification' else 'r2'):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=cv, scoring=scoring, n_jobs=-1
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train', color='steelblue')
    ax.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1),
                    train_scores.mean(1)+train_scores.std(1), alpha=0.2, color='steelblue')
    ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Validation', color='coral')
    ax.fill_between(train_sizes, val_scores.mean(1)-val_scores.std(1),
                    val_scores.mean(1)+val_scores.std(1), alpha=0.2, color='coral')
    ax.set_xlabel("Taille du training set")
    ax.set_ylabel(scoring)
    ax.set_title("Courbe d'apprentissage — Diagnostic Biais/Variance", fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("ℹ La courbe d'apprentissage sera tracée dans la section 5 après modélisation.")


### 3.2 Vraisemblance (MLE)

In [ ]:
# ── Maximum de Vraisemblance (rappel théorique) ───────────────────
print("""
L(θ | X) = P(X | θ)  ← probabilité des données sachant les paramètres

MLE : trouver θ qui maximise log L(θ|X) = Σ log P(xᵢ|θ)
→ La régression logistique utilise MLE pour estimer ses coefficients
""")


### 3.3 Théorème de Bayes

In [ ]:
print("""
P(θ | X) = P(X | θ) × P(θ) / P(X)
 Posterior  Vraisemblance   Prior   Evidence

Naive Bayes suppose l'indépendance des features :
P(y|x₁,...,xₙ) ∝ P(y) × P(x₁|y) × P(x₂|y) × ... × P(xₙ|y)
""")


### 3.4 Régularisation

In [ ]:
print("""
MÉTHODE      | PÉNALITÉ       | EFFET                        | USAGE
-------------|----------------|------------------------------|------------------
L1 (LASSO)   | λ Σ|βᵢ|        | Pousse coefficients vers 0   | Sélection features
L2 (Ridge)   | λ Σβᵢ²         | Réduit sans annuler          | Multicolinéarité
Elastic Net  | α·L1 + (1-α)·L2| Combine les deux             | Meilleur des deux
Dropout (DL) | Désactive neurones aléatoires | Évite co-adaptation | Réseaux DL
""")

# Démonstration Ridge vs LASSO (sur données simulées)
from sklearn.datasets import make_regression
X_sim, y_sim = make_regression(n_samples=100, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_sim, y_sim, test_size=0.2, random_state=42)

alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_coefs = [Ridge(alpha=a).fit(X_tr, y_tr).coef_ for a in alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=5000).fit(X_tr, y_tr).coef_ for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for coefs, ax, title, color in [(ridge_coefs, axes[0], 'Ridge (L2)', 'steelblue'),
                                  (lasso_coefs, axes[1], 'LASSO (L1)', 'coral')]:
    coefs = np.array(coefs)
    for j in range(coefs.shape[1]):
        ax.plot(np.log10(alphas), coefs[:, j], alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel("log₁₀(alpha)")
    ax.set_ylabel("Valeur des coefficients")
    ax.set_title(f"{title} — chemin de régularisation", fontweight='bold')
plt.tight_layout()
plt.show()
print("Ridge : coefficients réduits progressivement")
print("LASSO : coefficients mis à zéro (sélection de features)")


### 3.5 Validation croisée

In [ ]:
# ── Stratified K-Fold (standard) ─────────────────────────────────
N_SPLITS = 5   # 5 ou 10

if TASK == "classification":
    cv_strategy = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    print(f"✓ StratifiedKFold (k={N_SPLITS}) — préserve la distribution des classes")
else:
    from sklearn.model_selection import KFold
    cv_strategy = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    print(f"✓ KFold (k={N_SPLITS})")

print()
print("""
MÉTHODE           | DESCRIPTION                                  | QUAND
------------------|----------------------------------------------|------------------
K-Fold (k=5/10)   | k groupes, k itérations train/test           | Standard
Stratified K-Fold | Préserve distribution classes                | Classification ✓
LOO               | N folds pour N observations                   | Petits datasets
TimeSeriesSplit   | Folds chronologiques                         | Séries temporelles
Repeated K-Fold   | Répète le K-Fold plusieurs fois              | Réduire variance
""")


---
## 4. Preprocessing — Préparer les données

### 4.1 Normalisation / Standardisation


In [ ]:
print("""
MÉTHODE         | FORMULE              | RÉSULTAT       | USAGE
----------------|----------------------|----------------|---------------------
StandardScaler  | (x-μ)/σ              | μ=0, σ=1       | LR, SVM, PCA, KNN
MinMaxScaler    | (x-min)/(max-min)    | [0, 1]         | Réseaux de neurones
RobustScaler    | (x-médiane)/IQR      | Robuste outliers| Avec outliers
MaxAbsScaler    | x/max(|x|)           | [-1, 1]        | Données creuses
Aucune          | —                    | —              | RF, XGBoost (arbres)
""")

# Choix du scaler
SCALER_CHOICE = "standard"   # "standard", "minmax", "robust", "maxabs", "none"

scaler_map = {
    "standard": StandardScaler(),
    "minmax":   MinMaxScaler(),
    "robust":   RobustScaler(),
    "maxabs":   MaxAbsScaler(),
    "none":     "passthrough",
}
SCALER = scaler_map[SCALER_CHOICE]
print(f"\n✓ Scaler choisi : {SCALER_CHOICE} → {SCALER}")


### 4.2 Feature Engineering

In [ ]:
# ── Feature Engineering — adapter à votre dataset ────────────────
# Décommentez et adaptez les transformations pertinentes

df_fe = df.copy()

# Exemple 1 : Binarisation (ex : was_contacted_before pour bank dataset)
# df_fe['was_contacted_before'] = (df_fe['pdays'] != -1).astype(int)

# Exemple 2 : Transformation log (réduire asymétrie)
# for col in ['balance', 'income']:
#     if col in df_fe.columns:
#         df_fe[f'log_{col}'] = np.log1p(df_fe[col].clip(lower=0))

# Exemple 3 : Ratio / interaction
# if 'age' in df_fe.columns and 'campaign' in df_fe.columns:
#     df_fe['age_campaign_ratio'] = df_fe['age'] / (df_fe['campaign'] + 1)

# Exemple 4 : Flag valeur négative
# for col in num_cols:
#     if df_fe[col].min() < 0:
#         df_fe[f'neg_{col}'] = (df_fe[col] < 0).astype(int)

# Recalculer num_cols et cat_cols après FE
df = df_fe.copy()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_COL in num_cols:
    num_cols.remove(TARGET_COL)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if TARGET_COL in cat_cols:
    cat_cols.remove(TARGET_COL)

print(f"✓ Feature engineering appliqué")
print(f"  Variables numériques  : {len(num_cols)}")
print(f"  Variables catégorielles : {len(cat_cols)}")


### 4.3 Réduction de Dimensionnalité

In [ ]:
# ── PCA — Principal Component Analysis ────────────────────────────
PCA_ENABLED = False      # Passer à True pour activer
PCA_VARIANCE = 0.95      # conserver 95% de la variance

if PCA_ENABLED and len(num_cols) >= 3:
    from sklearn.preprocessing import StandardScaler as SS
    X_num_scaled = SS().fit_transform(df[num_cols].fillna(0))
    
    pca = PCA(n_components=PCA_VARIANCE)
    X_pca = pca.fit_transform(X_num_scaled)
    
    print(f"PCA : {len(num_cols)} → {pca.n_components_} composantes "
          f"(variance retenue : {pca.explained_variance_ratio_.sum():.1%})")
    
    # Scree plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(range(1, pca.n_components_+1), pca.explained_variance_ratio_, color='steelblue')
    axes[0].set_xlabel("Composante")
    axes[0].set_ylabel("Variance expliquée")
    axes[0].set_title("Scree Plot", fontweight='bold')
    
    axes[1].plot(range(1, pca.n_components_+1),
                 np.cumsum(pca.explained_variance_ratio_), 'o-', color='coral')
    axes[1].axhline(0.95, color='gray', linestyle='--', label='95%')
    axes[1].set_xlabel("Nombre de composantes")
    axes[1].set_ylabel("Variance cumulée")
    axes[1].set_title("Variance cumulée", fontweight='bold')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("ℹ PCA désactivée (PCA_ENABLED = False). Activer si nécessaire.")


In [ ]:
# ── t-SNE / UMAP — Visualisation 2D ──────────────────────────────
TSNE_ENABLED = False   # Lent — activer uniquement pour visualisation

if TSNE_ENABLED and len(num_cols) >= 2:
    from sklearn.preprocessing import StandardScaler as SS
    X_vis = SS().fit_transform(df[num_cols].fillna(0))
    
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    X_2d = tsne.fit_transform(X_vis[:min(2000, len(X_vis))])  # limiter à 2000 points
    
    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                         c=df[TARGET_COL].iloc[:len(X_2d)].astype('category').cat.codes,
                         cmap='tab10', alpha=0.6, s=10)
    plt.colorbar(scatter, ax=ax)
    ax.set_title("t-SNE — Visualisation 2D colorée par classe", fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("ℹ t-SNE désactivé (TSNE_ENABLED = False). Activer uniquement pour visualisation.")


---
## 5. Modèles de Classification

### 5.0 Préparation du dataset final


In [ ]:
# ── Split train / test ───────────────────────────────────────────
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Encoder la cible si nécessaire
if y.dtype == 'object':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), index=y.index, name=TARGET_COL)
    print(f"Cible encodée : {dict(zip(le.classes_, le.transform(le.classes_)))}")

TEST_SIZE = 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE,
    stratify=y if TASK == "classification" else None,
    random_state=42
)
print(f"✓ Train : {X_train.shape} | Test : {X_test.shape}")
print(f"  Distribution train : {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"  Distribution test  : {y_test.value_counts(normalize=True).round(3).to_dict()}")


In [ ]:
# ── Constructeur de Pipeline ─────────────────────────────────────
# Gère l'imputation + normalisation + encodage de façon sûre (pas de leakage)

def build_imputer(strategy_num, strategy_cat, constant='unknown'):
    """Retourne les imputeurs pour numériques et catégoriels."""
    if strategy_num == "knn":
        imp_num = KNNImputer(n_neighbors=5)
    elif strategy_num == "iterative":
        from sklearn.experimental import enable_iterative_imputer  # noqa
        from sklearn.impute import IterativeImputer
        imp_num = IterativeImputer(max_iter=10, random_state=42)
    else:
        imp_num = SimpleImputer(strategy=strategy_num)
    
    if strategy_cat == "constant":
        imp_cat = SimpleImputer(strategy='constant', fill_value=constant)
    else:
        imp_cat = SimpleImputer(strategy=strategy_cat)
    return imp_num, imp_cat

def build_preprocessor(num_feats, cat_feats, scaler="standard",
                        imp_num="median", imp_cat="most_frequent"):
    """Construit le ColumnTransformer complet."""
    imp_n, imp_c = build_imputer(imp_num, imp_cat)
    
    scaler_map = {
        "standard": StandardScaler(), "minmax": MinMaxScaler(),
        "robust": RobustScaler(), "none": "passthrough",
    }
    scl = scaler_map.get(scaler, StandardScaler())
    
    transformers = []
    if num_feats:
        transformers.append(('num', Pipeline([('imp', imp_n), ('scl', scl)]), num_feats))
    if cat_feats:
        transformers.append((
            'cat',
            Pipeline([
                ('imp', imp_c),
                ('enc', OneHotEncoder(drop='first', sparse_output=False,
                                      handle_unknown='ignore'))
            ]),
            cat_feats
        ))
    return ColumnTransformer(transformers, remainder='drop')

preprocessor = build_preprocessor(num_cols, cat_cols,
                                   scaler=SCALER_CHOICE,
                                   imp_num=IMPUTER_NUM,
                                   imp_cat=IMPUTER_CAT)
print("✓ Preprocessor construit")


### 5.1 Régression Logistique — Baseline

In [ ]:
# ── Régression Logistique ────────────────────────────────────────
cw = 'balanced' if (TASK == 'classification' and IMBALANCE_STRATEGY == 'class_weight') else None

lr_model = LogisticRegression(C=1.0, class_weight=cw, max_iter=1000, random_state=42)
pipe_lr = Pipeline([('pre', preprocessor), ('clf', lr_model)])
pipe_lr.fit(X_train, y_train)

y_pred_lr = pipe_lr.predict(X_test)
print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))

if TASK == "classification" and len(y.unique()) == 2:
    y_proba_lr = pipe_lr.predict_proba(X_test)[:, 1]
    auc_lr = roc_auc_score(y_test, y_proba_lr)
    print(f"AUC-ROC : {auc_lr:.4f}")
    
    # Odds ratios
    try:
        feature_names_out = pipe_lr.named_steps['pre'].get_feature_names_out()
        odds_ratios = np.exp(pipe_lr.named_steps['clf'].coef_[0])
        or_series = pd.Series(odds_ratios, index=feature_names_out).sort_values(ascending=False)
        print("\nTop 10 Odds Ratios :")
        print(or_series.head(10).round(3))
    except Exception as e:
        print(f"(Odds ratios non disponibles : {e})")


### 5.2 Support Vector Machine (SVM)

In [ ]:
# ── SVM ──────────────────────────────────────────────────────────
# ⚠ Toujours normaliser avant SVM (inclus dans le Pipeline)
SVM_ENABLED = True    # Peut être lent sur grands datasets — désactiver si nécessaire

if SVM_ENABLED and TASK == "classification":
    svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    pipe_svm = Pipeline([('pre', preprocessor), ('clf', svm_model)])
    pipe_svm.fit(X_train, y_train)
    y_pred_svm = pipe_svm.predict(X_test)
    print("=== SVM (kernel=rbf) ===")
    print(classification_report(y_test, y_pred_svm))
    if len(y.unique()) == 2:
        y_proba_svm = pipe_svm.predict_proba(X_test)[:, 1]
        print(f"AUC-ROC : {roc_auc_score(y_test, y_proba_svm):.4f}")
else:
    print("ℹ SVM désactivé ou tâche de régression.")


### 5.3 K-Nearest Neighbors (KNN)

In [ ]:
# ── KNN ──────────────────────────────────────────────────────────
# Distance de Minkowski : p=1 → Manhattan, p=2 → Euclidienne
if TASK == "classification":
    knn_model = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
    pipe_knn = Pipeline([('pre', preprocessor), ('clf', knn_model)])
    pipe_knn.fit(X_train, y_train)
    y_pred_knn = pipe_knn.predict(X_test)
    print("=== KNN (k=5, euclidean) ===")
    print(classification_report(y_test, y_pred_knn))
    if len(y.unique()) == 2:
        y_proba_knn = pipe_knn.predict_proba(X_test)[:, 1]
        print(f"AUC-ROC : {roc_auc_score(y_test, y_proba_knn):.4f}")


### 5.4 Arbre de Décision (CART)

In [ ]:
# ── Arbre de Décision ────────────────────────────────────────────
if TASK == "classification":
    tree_model = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20,
                                        criterion='gini', random_state=42)
    pipe_tree = Pipeline([('pre', preprocessor), ('clf', tree_model)])
    pipe_tree.fit(X_train, y_train)
    y_pred_tree = pipe_tree.predict(X_test)
    print("=== Decision Tree (max_depth=5) ===")
    print(classification_report(y_test, y_pred_tree))
    if len(y.unique()) == 2:
        y_proba_tree = pipe_tree.predict_proba(X_test)[:, 1]
        print(f"AUC-ROC : {roc_auc_score(y_test, y_proba_tree):.4f}")
    
    # Visualisation texte de l'arbre
    try:
        fn = pipe_tree.named_steps['pre'].get_feature_names_out()
        print("\nStructure de l'arbre (5 premières lignes) :")
        tree_text = export_text(pipe_tree.named_steps['clf'], feature_names=list(fn))
        print("\n".join(tree_text.split("\n")[:20]))
    except:
        pass


### 5.5 Random Forest

In [ ]:
# ── Random Forest ────────────────────────────────────────────────
cw = 'balanced' if (TASK == 'classification' and IMBALANCE_STRATEGY == 'class_weight') else None

if TASK == "classification":
    rf_model = RandomForestClassifier(n_estimators=200, max_depth=None,
                                       max_features='sqrt', class_weight=cw,
                                       n_jobs=-1, random_state=42)
else:
    rf_model = RandomForestRegressor(n_estimators=200, max_features='sqrt',
                                      n_jobs=-1, random_state=42)

pipe_rf = Pipeline([('pre', preprocessor), ('clf', rf_model)])
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

print("=== Random Forest ===")
if TASK == "classification":
    print(classification_report(y_test, y_pred_rf))
    if len(y.unique()) == 2:
        y_proba_rf = pipe_rf.predict_proba(X_test)[:, 1]
        print(f"AUC-ROC : {roc_auc_score(y_test, y_proba_rf):.4f}")
else:
    mse = mean_squared_error(y_test, y_pred_rf)
    print(f"RMSE : {np.sqrt(mse):.4f}")
    print(f"MAE  : {mean_absolute_error(y_test, y_pred_rf):.4f}")
    print(f"R²   : {r2_score(y_test, y_pred_rf):.4f}")

# Feature importance RF
try:
    fn = pipe_rf.named_steps['pre'].get_feature_names_out()
    fi = pd.Series(pipe_rf.named_steps['clf'].feature_importances_, index=fn)
    fi = fi.sort_values(ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(9, 6))
    fi.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.invert_yaxis()
    ax.set_title("Random Forest — Feature Importance (top 15)", fontweight='bold')
    ax.set_xlabel("Importance (réduction d'impureté)")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"(Feature importance non disponible : {e})")


### 5.6 Gradient Boosting (XGBoost / LightGBM / CatBoost)

In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier, XGBRegressor
    
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    spw = neg / pos if pos > 0 else 1
    
    if TASK == "classification":
        xgb_model = XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=spw if IMBALANCE_STRATEGY == 'class_weight' else 1,
            eval_metric='auc', random_state=42, verbosity=0, n_jobs=-1
        )
    else:
        xgb_model = XGBRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=0, n_jobs=-1
        )
    
    pipe_xgb = Pipeline([('pre', preprocessor), ('clf', xgb_model)])
    pipe_xgb.fit(X_train, y_train)
    y_pred_xgb = pipe_xgb.predict(X_test)
    
    print("=== XGBoost ===")
    if TASK == "classification":
        print(classification_report(y_test, y_pred_xgb))
        if len(y.unique()) == 2:
            y_proba_xgb = pipe_xgb.predict_proba(X_test)[:, 1]
            print(f"AUC-ROC : {roc_auc_score(y_test, y_proba_xgb):.4f}")
    else:
        mse = mean_squared_error(y_test, y_pred_xgb)
        print(f"RMSE : {np.sqrt(mse):.4f} | MAE : {mean_absolute_error(y_test,y_pred_xgb):.4f}"
              f" | R² : {r2_score(y_test,y_pred_xgb):.4f}")
              
except ImportError:
    print("ℹ XGBoost non installé. Exécuter : !pip install xgboost")


In [ ]:
# ── LightGBM (plus rapide que XGBoost sur grands datasets) ──────
try:
    import lightgbm as lgb
    
    if TASK == "classification":
        lgbm_model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                          num_leaves=31, n_jobs=-1, random_state=42,
                                          verbose=-1)
    else:
        lgbm_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                         num_leaves=31, n_jobs=-1, random_state=42,
                                         verbose=-1)
    
    pipe_lgbm = Pipeline([('pre', preprocessor), ('clf', lgbm_model)])
    pipe_lgbm.fit(X_train, y_train)
    y_pred_lgbm = pipe_lgbm.predict(X_test)
    
    print("=== LightGBM ===")
    if TASK == "classification":
        print(classification_report(y_test, y_pred_lgbm))
        if len(y.unique()) == 2:
            y_proba_lgbm = pipe_lgbm.predict_proba(X_test)[:, 1]
            print(f"AUC-ROC : {roc_auc_score(y_test, y_proba_lgbm):.4f}")
    else:
        mse = mean_squared_error(y_test, y_pred_lgbm)
        print(f"RMSE : {np.sqrt(mse):.4f} | R² : {r2_score(y_test,y_pred_lgbm):.4f}")
        
except ImportError:
    print("ℹ LightGBM non installé. Exécuter : !pip install lightgbm")


### 5.7 Naive Bayes

In [ ]:
# ── Naive Bayes ──────────────────────────────────────────────────
# GaussianNB : features continues (suppose distribution normale)
# MultinomialNB : comptage de mots (NLP)
# BernoulliNB : features binaires

if TASK == "classification":
    gnb_model = GaussianNB()
    pipe_gnb = Pipeline([('pre', preprocessor), ('clf', gnb_model)])
    pipe_gnb.fit(X_train, y_train)
    y_pred_gnb = pipe_gnb.predict(X_test)
    print("=== Gaussian Naive Bayes ===")
    print(classification_report(y_test, y_pred_gnb))
    if len(y.unique()) == 2:
        y_proba_gnb = pipe_gnb.predict_proba(X_test)[:, 1]
        print(f"AUC-ROC : {roc_auc_score(y_test, y_proba_gnb):.4f}")


---
## 6. Métriques d'Évaluation

### 6.1 & 6.2 Métriques de classification


In [ ]:
# ── Comparaison complète des modèles ─────────────────────────────
if TASK == "classification":
    # Référencer les modèles disponibles
    model_results = {}
    
    try:
        model_results['Logistic Regression'] = {
            'pred': y_pred_lr, 'proba': pipe_lr.predict_proba(X_test)[:,1]
        }
    except: pass
    try:
        if SVM_ENABLED:
            model_results['SVM (rbf)'] = {
                'pred': y_pred_svm, 'proba': pipe_svm.predict_proba(X_test)[:,1]
            }
    except: pass
    try:
        model_results['KNN'] = {
            'pred': y_pred_knn, 'proba': pipe_knn.predict_proba(X_test)[:,1]
        }
    except: pass
    try:
        model_results['Decision Tree'] = {
            'pred': y_pred_tree, 'proba': pipe_tree.predict_proba(X_test)[:,1]
        }
    except: pass
    try:
        model_results['Random Forest'] = {
            'pred': y_pred_rf, 'proba': pipe_rf.predict_proba(X_test)[:,1]
        }
    except: pass
    try:
        model_results['XGBoost'] = {
            'pred': y_pred_xgb, 'proba': pipe_xgb.predict_proba(X_test)[:,1]
        }
    except: pass
    try:
        model_results['LightGBM'] = {
            'pred': y_pred_lgbm, 'proba': pipe_lgbm.predict_proba(X_test)[:,1]
        }
    except: pass
    try:
        model_results['Naive Bayes'] = {
            'pred': y_pred_gnb, 'proba': pipe_gnb.predict_proba(X_test)[:,1]
        }
    except: pass
    
    # Tableau de comparaison
    binary = len(y.unique()) == 2
    rows = []
    for name, res in model_results.items():
        row = {
            'Modèle': name,
            'Accuracy': (res['pred'] == y_test).mean(),
            'F1 (macro)': f1_score(y_test, res['pred'], average='macro', zero_division=0),
        }
        if binary:
            row['AUC-ROC'] = roc_auc_score(y_test, res['proba'])
            row['Log-Loss'] = log_loss(y_test, res['proba'])
        rows.append(row)
    
    df_compare = pd.DataFrame(rows).set_index('Modèle').round(4)
    sort_col = 'AUC-ROC' if binary else 'F1 (macro)'
    df_compare = df_compare.sort_values(sort_col, ascending=False)
    print("=== COMPARAISON DES MODÈLES ===")
    print(df_compare.to_string())


In [ ]:
# ── Matrice de confusion — Meilleur modèle ───────────────────────
if TASK == "classification" and model_results:
    best_model_name = df_compare.index[0]
    best_preds = model_results[best_model_name]['pred']
    
    cm = confusion_matrix(y_test, best_preds)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                linewidths=0.5)
    axes[0].set_xlabel("Prédit")
    axes[0].set_ylabel("Réel")
    axes[0].set_title(f"Matrice de confusion — {best_model_name}", fontweight='bold')
    
    # Courbe ROC de tous les modèles
    if binary:
        for name, res in model_results.items():
            from sklearn.metrics import roc_curve
            fpr, tpr, _ = roc_curve(y_test, res['proba'])
            auc_val = roc_auc_score(y_test, res['proba'])
            axes[1].plot(fpr, tpr, lw=1.5, label=f'{name} (AUC={auc_val:.3f})')
        axes[1].plot([0,1],[0,1],'k--', alpha=0.4, label='Aléatoire')
        axes[1].set_xlabel("Taux Faux Positifs (FPR)")
        axes[1].set_ylabel("Taux Vrais Positifs (TPR = Recall)")
        axes[1].set_title("Courbes ROC — Comparaison", fontweight='bold')
        axes[1].legend(fontsize=8)
        axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Courbe Precision-Recall (déséquilibre de classes) ────────────
if TASK == "classification" and binary:
    from sklearn.metrics import precision_recall_curve, average_precision_score
    
    fig, ax = plt.subplots(figsize=(9, 6))
    for name, res in model_results.items():
        precision, recall, _ = precision_recall_curve(y_test, res['proba'])
        ap = average_precision_score(y_test, res['proba'])
        ax.plot(recall, precision, lw=1.5, label=f'{name} (AP={ap:.3f})')
    
    baseline = y_test.mean()
    ax.axhline(baseline, color='black', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Courbes Precision-Recall (meilleure pour données déséquilibrées)", fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("ℹ La courbe Precision-Recall est plus informative que ROC pour les classes très déséquilibrées.")


### 6.3 Métriques de régression

In [ ]:
if TASK == "regression":
    reg_results = {}
    for name, pipe in [("RF", pipe_rf)]:
        try:
            pred = pipe.predict(X_test)
            reg_results[name] = {
                'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
                'MAE':  mean_absolute_error(y_test, pred),
                'R²':   r2_score(y_test, pred),
                'pred': pred
            }
        except: pass
    
    if reg_results:
        df_reg = pd.DataFrame(reg_results).T.round(4)
        print("=== MÉTRIQUES DE RÉGRESSION ===")
        print(df_reg)
        
        # Scatter Réel vs Prédit
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        for i, (name, res) in enumerate(reg_results.items()):
            if i < 2:
                axes[i].scatter(y_test, res['pred'], alpha=0.4, s=10, color='steelblue')
                lims = [min(y_test.min(), res['pred'].min()), max(y_test.max(), res['pred'].max())]
                axes[i].plot(lims, lims, 'r--', lw=1.5, label='Parfait')
                axes[i].set_xlabel("Valeurs réelles")
                axes[i].set_ylabel("Valeurs prédites")
                axes[i].set_title(f"{name} — Réel vs Prédit", fontweight='bold')
                axes[i].legend()
        plt.tight_layout()
        plt.show()


---
## 7. Modèles de Régression

> Activés automatiquement si TASK = "regression"


In [ ]:
# ── Régression Linéaire (OLS) ────────────────────────────────────
if TASK == "regression":
    pipe_ols = Pipeline([('pre', preprocessor), ('reg', LinearRegression())])
    pipe_ols.fit(X_train, y_train)
    y_pred_ols = pipe_ols.predict(X_test)
    
    print("=== Régression Linéaire (OLS) ===")
    print(f"R²   : {r2_score(y_test, y_pred_ols):.4f}")
    print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_ols)):.4f}")
    print(f"MAE  : {mean_absolute_error(y_test, y_pred_ols):.4f}")
    
    # Analyse des résidus
    residuals = y_test - y_pred_ols
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_pred_ols, residuals, alpha=0.4, s=10, color='steelblue')
    axes[0].axhline(0, color='red', linestyle='--')
    axes[0].set_xlabel("Valeurs prédites")
    axes[0].set_ylabel("Résidus")
    axes[0].set_title("Résidus vs Prédits (Homoscédasticité)", fontweight='bold')
    
    stats.probplot(residuals, dist='norm', plot=axes[1])
    axes[1].set_title("Q-Q Plot des résidus (Normalité)", fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("ℹ Section régression désactivée (TASK = 'classification').")


In [ ]:
# ── LASSO, Ridge, Elastic Net ────────────────────────────────────
if TASK == "regression":
    for name, reg in [
        ("Ridge (L2)",     RidgeCV(alphas=[0.01, 0.1, 1, 10, 100])),
        ("LASSO (L1)",     LassoCV(cv=5, max_iter=5000)),
        ("Elastic Net",    ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5, max_iter=5000)),
    ]:
        pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        print(f"{name:20s} → R²={r2_score(y_test, pred):.4f}  "
              f"RMSE={np.sqrt(mean_squared_error(y_test,pred)):.4f}")


In [ ]:
# ── Régression Polynomiale ────────────────────────────────────────
if TASK == "regression" and len(num_cols) > 0:
    POLY_DEGREE = 2   # degré du polynôme
    poly_pipe = Pipeline([
        ('pre', preprocessor),
        ('poly', PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)),
        ('scl', StandardScaler()),
        ('reg', LinearRegression())
    ])
    poly_pipe.fit(X_train, y_train)
    pred_poly = poly_pipe.predict(X_test)
    print(f"Régression polynomiale (deg={POLY_DEGREE}) :"
          f" R²={r2_score(y_test, pred_poly):.4f}")


In [ ]:
# ── Gaussian Process Regression (GPR) ────────────────────────────
# ⚠ Complexité O(n³) — uniquement pour petits datasets (< 2000)
GPR_ENABLED = False

if TASK == "regression" and GPR_ENABLED and len(X_train) < 2000:
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel
    
    kernel = RBF(length_scale=1.0) + WhiteKernel(noise_level=0.1)
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=42)
    
    # GPR nécessite des features déjà transformées
    pre_fit = preprocessor.fit(X_train)
    X_tr_gpr = pre_fit.transform(X_train)
    X_te_gpr = pre_fit.transform(X_test)
    
    gpr.fit(X_tr_gpr, y_train)
    y_mean, y_std = gpr.predict(X_te_gpr, return_std=True)
    
    print(f"GPR : R²={r2_score(y_test, y_mean):.4f}")
    print(f"Incertitude moyenne (std) : {y_std.mean():.4f}")
else:
    if TASK == "regression":
        print("ℹ GPR désactivé (GPR_ENABLED=False). Activer uniquement pour petits datasets.")


---
## 8. Deep Learning — Réseaux de Neurones

> Nécessite PyTorch. Activer DL_ENABLED = True.


In [ ]:
DL_ENABLED = False    # Passer à True si PyTorch est installé

if DL_ENABLED:
    try:
        import torch
        import torch.nn as nn
        import torch.optim as optim
        print(f"✓ PyTorch {torch.__version__}")
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"  Device : {device}")
    except ImportError:
        print("ℹ PyTorch non installé. Exécuter : !pip install torch torchvision")
        DL_ENABLED = False
else:
    print("ℹ Deep Learning désactivé (DL_ENABLED = False).")


### 8.1 Fonctions d'activation

In [ ]:
# ── Visualisation des fonctions d'activation ─────────────────────
z = np.linspace(-4, 4, 200)

activations = {
    'Sigmoid σ(z)':    1 / (1 + np.exp(-z)),
    'Tanh':            np.tanh(z),
    'ReLU':            np.maximum(0, z),
    'Leaky ReLU':      np.where(z > 0, z, 0.01 * z),
    'GELU':            z * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3))),
}

fig, axes = plt.subplots(1, len(activations), figsize=(16, 3))
colors = ['steelblue', 'coral', 'green', 'purple', 'orange']
for ax, (name, vals), color in zip(axes, activations.items(), colors):
    ax.plot(z, vals, color=color, lw=2)
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.set_ylim(-1.5, 1.5)
    ax.grid(alpha=0.3)
plt.suptitle("Fonctions d'activation", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 8.2 MLP (réseau dense) avec PyTorch

In [ ]:
# ── MLP PyTorch ──────────────────────────────────────────────────
if DL_ENABLED:
    import torch, torch.nn as nn, torch.optim as optim
    
    # Préparer les tenseurs
    pre_fit = preprocessor.fit(X_train)
    X_tr_np = pre_fit.transform(X_train).astype(np.float32)
    X_te_np = pre_fit.transform(X_test).astype(np.float32)
    y_tr_np = y_train.values.astype(np.float32)
    y_te_np = y_test.values.astype(np.float32)
    
    X_tr_t = torch.tensor(X_tr_np)
    X_te_t = torch.tensor(X_te_np)
    y_tr_t = torch.tensor(y_tr_np)
    
    # Architecture MLP
    class MLP(nn.Module):
        def __init__(self, input_dim, hidden_dims, output_dim):
            super().__init__()
            layers = []
            prev = input_dim
            for h in hidden_dims:
                layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.3)]
                prev = h
            layers.append(nn.Linear(prev, output_dim))
            self.net = nn.Sequential(*layers)
        def forward(self, x):
            return self.net(x)
    
    input_dim  = X_tr_np.shape[1]
    output_dim = 1 if TASK == "classification" and len(y.unique()) == 2 else len(y.unique()) if TASK == "classification" else 1
    
    model_mlp  = MLP(input_dim, [128, 64, 32], output_dim)
    optimizer  = optim.Adam(model_mlp.parameters(), lr=1e-3)
    criterion  = nn.BCEWithLogitsLoss() if TASK == "classification" and output_dim == 1 else nn.CrossEntropyLoss()
    
    N_EPOCHS_MLP = 50
    train_losses = []
    
    for epoch in range(N_EPOCHS_MLP):
        model_mlp.train()
        optimizer.zero_grad()
        out = model_mlp(X_tr_t).squeeze()
        loss = criterion(out, y_tr_t)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(train_losses, color='steelblue')
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (BCEWithLogitsLoss)")
    ax.set_title("Courbe de perte — MLP", fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"✓ MLP entraîné ({N_EPOCHS_MLP} epochs) | Loss finale : {train_losses[-1]:.4f}")
else:
    print("ℹ MLP désactivé (DL_ENABLED = False).")


### 8.5 CNN, 8.6 Transfer Learning, 8.7 LSTM

In [ ]:
# ── Rappels architectures CNN / Transfer Learning / LSTM ─────────
print("""
CNN (images) — Couches principales :
  Conv2D → BatcnNorm → ReLU → MaxPool → Dropout → Flatten → Dense

Transfer Learning — Approche :
  1. Charger modèle pré-entraîné (ResNet50, VGG16, BERT...)
  2. Geler les couches (feature extraction) OU fine-tuner tout
  3. Remplacer la tête de classification

Architecture CNN (PyTorch) : voir document section 8.5
Architecture LSTM (PyTorch) : voir document section 8.7

→ Activer DL_ENABLED = True et adapter le code selon votre tâche.
""")


---
## 9. Traitement du Langage Naturel (NLP)


In [ ]:
# ── Pipeline NLP de base ─────────────────────────────────────────
NLP_ENABLED = False   # Activer si vous avez une colonne texte

if NLP_ENABLED:
    TEXT_COL = "text"   # ← nom de la colonne texte
    
    # Preprocessing
    import re
    try:
        import nltk
        nltk.download('punkt', quiet=True)
        nltk.download('stopwords', quiet=True)
        nltk.download('wordnet', quiet=True)
        from nltk.tokenize import word_tokenize
        from nltk.corpus import stopwords
        from nltk.stem import WordNetLemmatizer
        
        lemmatizer = WordNetLemmatizer()
        stop_words = set(stopwords.words('french'))
        
        def preprocess_text(text):
            tokens = word_tokenize(str(text).lower())
            tokens = [lemmatizer.lemmatize(t)
                      for t in tokens if t.isalpha() and t not in stop_words]
            return ' '.join(tokens)
        
        df['text_clean'] = df[TEXT_COL].apply(preprocess_text)
        print("✓ Preprocessing NLP appliqué")
    except ImportError:
        print("ℹ NLTK non installé : !pip install nltk")
else:
    print("ℹ NLP désactivé (NLP_ENABLED = False).")


In [ ]:
# ── TF-IDF + Classification de texte ─────────────────────────────
if NLP_ENABLED:
    try:
        tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
        X_tfidf = tfidf.fit_transform(df['text_clean'])
        print(f"TF-IDF : matrice {X_tfidf.shape}")
        
        # Classification avec LR sur TF-IDF
        from sklearn.linear_model import LogisticRegression
        X_tr_nl, X_te_nl, y_tr_nl, y_te_nl = train_test_split(
            X_tfidf, df[TARGET_COL], test_size=0.2, random_state=42
        )
        lr_nlp = LogisticRegression(max_iter=1000)
        lr_nlp.fit(X_tr_nl, y_tr_nl)
        print(classification_report(y_te_nl, lr_nlp.predict(X_te_nl)))
    except Exception as e:
        print(f"Erreur : {e}")
else:
    print("""
REPRÉSENTATIONS VECTORIELLES NLP :
  BoW      : vecteur de fréquences de mots — simple, ignore l'ordre
  TF-IDF   : pondère les mots rares — bon pour recherche d'info
  Word2Vec : embedding dense contextuel — capture relations sémantiques
  BERT     : Transformer bidirectionnel — état de l'art, lourd
  """)


---
## 10. Big Data — Traitement à Grande Échelle


In [ ]:
# ── Les 5V du Big Data ───────────────────────────────────────────
print("""
DIMENSION  | DESCRIPTION                        | EXEMPLE
-----------|------------------------------------|--------------------------
Volume     | Quantité (TB, PB)                  | Logs serveur, transactions
Vélocité   | Vitesse de génération              | Flux Twitter, IoT temps réel
Variété    | Formats (structuré, non-structuré) | CSV, JSON, images, vidéos
Véracité   | Qualité et fiabilité               | Données manquantes, erreurs
Valeur     | Pertinence métier                  | Toutes données ≠ utiles
""")


In [ ]:
# ── Dask — Pandas à grande échelle ───────────────────────────────
DASK_ENABLED = False   # Activer si dataset > RAM

if DASK_ENABLED:
    try:
        import dask.dataframe as dd
        # ddf = dd.read_csv('data_*.csv')
        # result = ddf.groupby('col')['num'].mean().compute()
        print("ℹ Dask disponible. Exemple :")
        print("  ddf = dd.read_csv('data_*.csv')")
        print("  result = ddf.groupby('col')['num'].mean().compute()")
    except ImportError:
        print("ℹ Dask non installé : !pip install dask[dataframe]")
else:
    print("ℹ Dask désactivé. Activer si dataset > mémoire disponible.")


In [ ]:
# ── Apache Spark (PySpark) ───────────────────────────────────────
SPARK_ENABLED = False   # Activer pour vraies données Big Data

if SPARK_ENABLED:
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.appName('DS52').getOrCreate()
        print(f"✓ Spark {spark.version}")
    except ImportError:
        print("ℹ PySpark non installé : !pip install pyspark")
else:
    print("""
ℹ Spark désactivé.

API PySpark — rappels :
  df = spark.read.csv('s3://...', header=True, inferSchema=True)
  df.filter(df['age'] > 30).groupBy('job').mean('balance').show()
  
Architectures Big Data :
  Ingestion  : Kafka, Kinesis
  Stockage   : HDFS, S3, Delta Lake
  Processing : Spark (batch), Kafka Streams (stream)
  Warehouse  : BigQuery, Redshift, Snowflake
  ML Scale   : MLflow, Ray, Horovod
  Orchestration : Airflow, Prefect
  """)


In [ ]:
# ── Optimisations mémoire scikit-learn ────────────────────────────
print("=== OPTIMISATIONS MÉMOIRE ===")
print()

# Réduire la mémoire des types
print("Avant optimisation :")
print(f"  {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

df_opt = df.copy()
for col in df_opt.select_dtypes(include=['int64']).columns:
    col_min, col_max = df_opt[col].min(), df_opt[col].max()
    if col_min >= -128 and col_max <= 127:
        df_opt[col] = df_opt[col].astype('int8')
    elif col_min >= -32768 and col_max <= 32767:
        df_opt[col] = df_opt[col].astype('int16')
    elif col_min >= -2147483648 and col_max <= 2147483647:
        df_opt[col] = df_opt[col].astype('int32')

for col in df_opt.select_dtypes(include=['float64']).columns:
    df_opt[col] = df_opt[col].astype('float32')

print("Après optimisation :")
print(f"  {df_opt.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()
print("Partial fit (out-of-core) :")
print("  SGDClassifier(loss='log_loss').partial_fit(X_chunk, y_chunk, classes=[0,1])")
print()
print("Parquet (format colonnaire) :")
print("  df.to_parquet('data.parquet')  # 10-100x moins de mémoire que CSV")


---
## 11. XAI — Explicabilité des Modèles

### 11.1 Pourquoi l'explicabilité ?


In [ ]:
print("""
Les modèles complexes (RF, XGBoost, réseaux) sont des "boîtes noires".
L'XAI permet de comprendre et d'auditer leurs décisions.
Obligation légale (RGPD) pour les décisions automatisées impactant des personnes.

Méthodes disponibles :
  1. Feature Importance par impureté (RF, XGBoost) — rapide, biaisé
  2. Permutation Importance — moins biaisée
  3. SHAP — contribution marginale par prédiction (théorie des jeux)
  4. LIME — modèle linéaire local autour d'une instance
  5. PDP / ICE — effet marginal moyen d'une feature
  6. Grad-CAM — pour les CNN sur images
""")


### 11.2 Feature Importance

In [ ]:
# ── Feature Importance par impureté vs Permutation ───────────────
try:
    fn = pipe_rf.named_steps['pre'].get_feature_names_out()
    
    # Importance par impureté (rapide, mais biaisée vers variables à haute cardinalité)
    fi_imp = pd.Series(pipe_rf.named_steps['clf'].feature_importances_, index=fn)
    fi_imp = fi_imp.sort_values(ascending=False).head(15)
    
    # Permutation Importance (moins biaisée, plus lente)
    X_test_pre = pipe_rf.named_steps['pre'].transform(X_test)
    r = permutation_importance(pipe_rf.named_steps['clf'], X_test_pre, y_test,
                                n_repeats=10, random_state=42, n_jobs=-1)
    fi_perm = pd.Series(r.importances_mean, index=fn).sort_values(ascending=False).head(15)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fi_imp.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].invert_yaxis()
    axes[0].set_title("Importance par impureté (rapide, biaisée)", fontweight='bold')
    
    fi_perm.plot(kind='barh', ax=axes[1], color='coral', edgecolor='white')
    axes[1].invert_yaxis()
    axes[1].set_title("Permutation Importance (moins biaisée)", fontweight='bold')
    
    plt.suptitle("Feature Importance — Comparaison", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Feature importance non disponible : {e}")


### 11.3 SHAP

In [ ]:
# ── SHAP — SHapley Additive exPlanations ────────────────────────
SHAP_ENABLED = False   # Activer après installation : !pip install shap

if SHAP_ENABLED:
    try:
        import shap
        shap.initjs()
        
        # TreeExplainer (optimisé pour RF, XGBoost, LightGBM)
        try:
            explainer = shap.TreeExplainer(pipe_xgb.named_steps['clf'])
            X_test_pre = pipe_xgb.named_steps['pre'].transform(X_test)
        except:
            explainer = shap.TreeExplainer(pipe_rf.named_steps['clf'])
            X_test_pre = pipe_rf.named_steps['pre'].transform(X_test)
        
        fn = pipe_rf.named_steps['pre'].get_feature_names_out()
        X_test_df = pd.DataFrame(X_test_pre[:200], columns=fn)
        shap_values = explainer.shap_values(X_test_df)
        
        # Si classification binaire, shap_values est une liste [class0, class1]
        sv = shap_values[1] if isinstance(shap_values, list) else shap_values
        
        # Summary plot (importance globale)
        print("=== SHAP Summary Plot ===")
        shap.summary_plot(sv, X_test_df, feature_names=list(fn))
        
        # Explication locale (1 prédiction)
        print("\n=== SHAP Waterfall — 1ère instance ===")
        shap.waterfall_plot(shap.Explanation(
            values=sv[0], base_values=explainer.expected_value
                         if not isinstance(explainer.expected_value, list)
                         else explainer.expected_value[1],
            data=X_test_df.iloc[0],
            feature_names=list(fn)
        ))
        
    except ImportError:
        print("ℹ SHAP non installé : !pip install shap")
else:
    print("ℹ SHAP désactivé. Activer SHAP_ENABLED = True après installation.")


### 11.4 LIME

In [ ]:
# ── LIME ─────────────────────────────────────────────────────────
LIME_ENABLED = False   # !pip install lime

if LIME_ENABLED:
    try:
        from lime.lime_tabular import LimeTabularExplainer
        
        pre_fit = preprocessor.fit(X_train)
        X_tr_lime = pre_fit.transform(X_train)
        X_te_lime = pre_fit.transform(X_test)
        fn = pre_fit.get_feature_names_out()
        
        explainer_lime = LimeTabularExplainer(
            X_tr_lime,
            feature_names=list(fn),
            class_names=[str(c) for c in y.unique()],
            mode='classification' if TASK == 'classification' else 'regression'
        )
        
        # Expliquer la 1ère instance du test set
        clf = pipe_rf.named_steps['clf']
        exp = explainer_lime.explain_instance(
            X_te_lime[0], clf.predict_proba, num_features=10
        )
        print("LIME — 1ère instance (feature, weight) :")
        for feat, weight in exp.as_list():
            print(f"  {feat:40s} : {weight:+.4f}")
        # exp.show_in_notebook()  # décommenter en Jupyter
        
    except ImportError:
        print("ℹ LIME non installé : !pip install lime")
else:
    print("ℹ LIME désactivé. Activer LIME_ENABLED = True.")


### 11.5 PDP et ICE

In [ ]:
# ── Partial Dependence Plot (PDP) & ICE ──────────────────────────
try:
    fn = list(pipe_rf.named_steps['pre'].get_feature_names_out())
    
    # Sélectionner les 2 features les plus importantes pour le PDP
    fi = pd.Series(pipe_rf.named_steps['clf'].feature_importances_, index=fn)
    top2 = fi.nlargest(2).index.tolist()
    top2_idx = [fn.index(f) for f in top2]
    
    pre_fit = pipeline_preprocessor = preprocessor.fit(X_train)
    X_train_pre = pd.DataFrame(pre_fit.transform(X_train), columns=fn)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # PDP individuels
    for i, (feat, ax) in enumerate(zip(top2, axes)):
        PartialDependenceDisplay.from_estimator(
            pipe_rf.named_steps['clf'], X_train_pre,
            features=[fn.index(feat)], feature_names=fn,
            kind='both',  # PDP + ICE
            ax=ax
        )
        ax.set_title(f"PDP + ICE — {feat}", fontweight='bold')
    
    plt.suptitle("Partial Dependence Plots", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"PDP non disponible : {e}")


---
## 12. Clustering & Détection d'Anomalies


In [ ]:
# ── K-Means — méthode du coude + silhouette ──────────────────────
CLUSTERING_ENABLED = True

if CLUSTERING_ENABLED and len(num_cols) >= 2:
    from sklearn.preprocessing import StandardScaler as SS
    X_clust = SS().fit_transform(df[num_cols].fillna(df[num_cols].median()))
    
    K_MAX = min(10, len(df) // 50)  # Limiter k
    inertias, sil_scores = [], []
    k_range = range(2, K_MAX + 1)
    
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_clust)
        inertias.append(km.inertia_)
        sil_scores.append(silhouette_score(X_clust, labels))
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(k_range, inertias, 'o-', color='steelblue')
    axes[0].set_xlabel("Nombre de clusters k")
    axes[0].set_ylabel("Inertie")
    axes[0].set_title("Méthode du coude", fontweight='bold')
    
    axes[1].plot(k_range, sil_scores, 'o-', color='coral')
    axes[1].set_xlabel("Nombre de clusters k")
    axes[1].set_ylabel("Score de silhouette")
    axes[1].set_title("Score de silhouette (max = meilleur)", fontweight='bold')
    
    best_k = k_range[np.argmax(sil_scores)]
    axes[1].axvline(best_k, color='green', linestyle='--', label=f'k optimal = {best_k}')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    
    print(f"✓ k optimal (silhouette) : {best_k}")
else:
    print("ℹ Clustering désactivé ou pas assez de variables numériques.")


In [ ]:
# ── K-Means final ────────────────────────────────────────────────
if CLUSTERING_ENABLED and len(num_cols) >= 2:
    K_FINAL = best_k   # Ou fixer manuellement : K_FINAL = 4
    
    km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
    cluster_labels = km_final.fit_predict(X_clust)
    df['cluster'] = cluster_labels
    
    print(f"=== K-Means (k={K_FINAL}) ===")
    print(df.groupby('cluster')[num_cols].mean().round(2))
    
    # Visualisation 2D (2 premières features numériques)
    if len(num_cols) >= 2:
        fig, ax = plt.subplots(figsize=(9, 6))
        scatter = ax.scatter(X_clust[:, 0], X_clust[:, 1], c=cluster_labels,
                             cmap='tab10', alpha=0.5, s=10)
        centers = km_final.cluster_centers_
        ax.scatter(centers[:, 0], centers[:, 1], c='black', s=150, marker='X', zorder=5)
        plt.colorbar(scatter, ax=ax)
        ax.set_xlabel(num_cols[0])
        ax.set_ylabel(num_cols[1])
        ax.set_title(f"K-Means (k={K_FINAL}) — 2 premières features", fontweight='bold')
        plt.tight_layout()
        plt.show()


In [ ]:
# ── Clustering Hiérarchique ───────────────────────────────────────
if CLUSTERING_ENABLED and len(num_cols) >= 2:
    import scipy.cluster.hierarchy as sch
    
    # Dendrogramme (sur un sous-ensemble pour la lisibilité)
    n_dendro = min(300, len(X_clust))
    X_dendro = X_clust[:n_dendro]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    sch.dendrogram(sch.linkage(X_dendro, method='ward'), ax=ax,
                   truncate_mode='lastp', p=20, leaf_rotation=45)
    ax.set_title("Dendrogramme — Clustering Hiérarchique (Ward)", fontweight='bold')
    ax.set_xlabel("Observations")
    ax.set_ylabel("Distance")
    plt.tight_layout()
    plt.show()
    
    ac = AgglomerativeClustering(n_clusters=K_FINAL, linkage='ward')
    ac_labels = ac.fit_predict(X_clust)
    sil_ac = silhouette_score(X_clust, ac_labels)
    sil_km = silhouette_score(X_clust, cluster_labels)
    print(f"Silhouette K-Means    : {sil_km:.4f}")
    print(f"Silhouette Hiérarchique : {sil_ac:.4f}")


In [ ]:
# ── DBSCAN (density-based, forme arbitraire) ─────────────────────
if CLUSTERING_ENABLED and len(num_cols) >= 2:
    EPS = 0.5         # rayon de voisinage
    MIN_SAMPLES = 10  # points minimum pour former un cluster
    
    db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
    db_labels = db.fit_predict(X_clust)
    
    n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = (db_labels == -1).sum()
    
    print(f"DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES})")
    print(f"  Clusters trouvés : {n_clusters_db}")
    print(f"  Points bruit (outliers) : {n_noise} ({n_noise/len(db_labels)*100:.1f}%)")
    
    if n_clusters_db > 1:
        mask = db_labels != -1
        sil_db = silhouette_score(X_clust[mask], db_labels[mask]) if mask.sum() > 1 else None
        if sil_db:
            print(f"  Silhouette (sans bruit) : {sil_db:.4f}")


In [ ]:
# ── Détection d'Anomalies ────────────────────────────────────────
if len(num_cols) >= 2:
    X_anom = df[num_cols].fillna(df[num_cols].median())
    from sklearn.preprocessing import StandardScaler as SS
    X_anom_sc = SS().fit_transform(X_anom)
    
    # Isolation Forest
    iso = IsolationForest(contamination=0.05, random_state=42)
    iso_labels = iso.fit_predict(X_anom_sc)
    n_iso = (iso_labels == -1).sum()
    
    # Local Outlier Factor
    lof = LocalOutlierFactor(n_neighbors=20)
    lof_labels = lof.fit_predict(X_anom_sc)
    n_lof = (lof_labels == -1).sum()
    
    print(f"Isolation Forest : {n_iso} anomalies ({n_iso/len(X_anom_sc)*100:.1f}%)")
    print(f"Local Outlier Factor : {n_lof} anomalies ({n_lof/len(X_anom_sc)*100:.1f}%)")
    
    # Visualisation sur les 2 premières features
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, labels, title, color in [
        (axes[0], iso_labels, "Isolation Forest", 'steelblue'),
        (axes[1], lof_labels, "Local Outlier Factor", 'coral')
    ]:
        normal = labels == 1
        ax.scatter(X_anom_sc[normal, 0], X_anom_sc[normal, 1],
                   c=color, alpha=0.4, s=8, label='Normal')
        ax.scatter(X_anom_sc[~normal, 0], X_anom_sc[~normal, 1],
                   c='red', s=30, marker='x', zorder=5, label='Anomalie')
        ax.set_title(title, fontweight='bold')
        ax.legend()
    plt.suptitle("Détection d'Anomalies", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


---
## 13. Séries Temporelles


In [ ]:
TS_ENABLED = False     # Activer si vous avez une série temporelle
DATE_COL  = None       # Colonne de dates, ex : "date"
VALUE_COL = None       # Colonne de valeurs, ex : "sales"

if TS_ENABLED and DATE_COL and VALUE_COL:
    df_ts = df[[DATE_COL, VALUE_COL]].copy()
    df_ts[DATE_COL] = pd.to_datetime(df_ts[DATE_COL])
    df_ts = df_ts.set_index(DATE_COL).sort_index()
    
    # Test de stationnarité (ADF)
    from statsmodels.tsa.stattools import adfuller
    adf_result = adfuller(df_ts[VALUE_COL].dropna())
    print(f"Test ADF — p-value : {adf_result[1]:.4f}")
    print(f"Série {'stationnaire' if adf_result[1] < 0.05 else 'NON stationnaire'} (seuil 5%)")
    
    # Décomposition STL
    from statsmodels.tsa.seasonal import seasonal_decompose
    decomp = seasonal_decompose(df_ts[VALUE_COL].dropna(), model='additive', period=12)
    fig = decomp.plot()
    fig.suptitle("Décomposition STL (trend + seasonal + residual)", fontweight='bold')
    plt.tight_layout()
    plt.show()
    
else:
    print("""
ℹ Séries temporelles désactivées. Configurer TS_ENABLED, DATE_COL, VALUE_COL.

Modèles disponibles :
  ARIMA(p,d,q)   : statsmodels.tsa.arima.model.ARIMA
  SARIMA         : statsmodels.tsa.statespace.sarimax.SARIMAX
  Auto-ARIMA     : pmdarima.auto_arima
  Prophet        : prophet.Prophet (Facebook)
  LSTM           : PyTorch (voir section 8.7)
  """)


In [ ]:
# ── ARIMA / SARIMA ────────────────────────────────────────────────
if TS_ENABLED and DATE_COL and VALUE_COL:
    from statsmodels.tsa.arima.model import ARIMA
    
    train_size = int(len(df_ts) * 0.8)
    train_ts, test_ts = df_ts.iloc[:train_size], df_ts.iloc[train_size:]
    
    # Ajustement ARIMA
    arima = ARIMA(train_ts[VALUE_COL], order=(2, 1, 2))
    arima_fit = arima.fit()
    forecast = arima_fit.forecast(steps=len(test_ts))
    
    from sklearn.metrics import mean_absolute_error
    mae_arima = mean_absolute_error(test_ts[VALUE_COL], forecast)
    print(f"ARIMA(2,1,2) — MAE : {mae_arima:.4f}")
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(train_ts.index, train_ts[VALUE_COL], label='Train', color='steelblue')
    ax.plot(test_ts.index, test_ts[VALUE_COL], label='Test', color='coral')
    ax.plot(test_ts.index, forecast, label='ARIMA forecast', color='green', linestyle='--')
    ax.legend()
    ax.set_title("ARIMA — Prévisions", fontweight='bold')
    plt.tight_layout()
    plt.show()


---
## 14. Optimisation des Hyperparamètres


In [ ]:
print("""
MÉTHODE              | DESCRIPTION                    | QUAND UTILISER
---------------------|--------------------------------|------------------
Grid Search          | Toutes les combinaisons        | ≤ 3 hyperparamètres
Randomized Search    | Échantillonnage aléatoire      | Grand espace — TOUJOURS préférer
Bayesian (Optuna)    | Modèle probabiliste guide      | Évaluation coûteuse (DL)
Halving Search       | Élimine les mauvais candidats  | Compromis rapidité/qualité
""")


In [ ]:
# ── RandomizedSearchCV (recommandé) ──────────────────────────────
from scipy.stats import randint, uniform

# Espace de recherche pour Random Forest
param_dist_rf = {
    'clf__n_estimators':     randint(100, 500),
    'clf__max_depth':        [None, 5, 10, 20, 30],
    'clf__min_samples_leaf': randint(5, 50),
    'clf__max_features':     ['sqrt', 'log2', None],
}

# Pipeline à optimiser (adapter selon TASK)
if TASK == "classification":
    base_clf = RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)
else:
    base_clf = RandomForestRegressor(n_jobs=-1, random_state=42)

pipe_search = Pipeline([('pre', preprocessor), ('clf', base_clf)])

scoring_metric = 'roc_auc' if TASK == 'classification' and len(y.unique()) == 2 else                  'f1_macro' if TASK == 'classification' else 'r2'

search = RandomizedSearchCV(
    pipe_search,
    param_distributions=param_dist_rf,
    n_iter=30,           # Augmenter à 100 pour un vrai tuning
    cv=StratifiedKFold(5, shuffle=True, random_state=42) if TASK == 'classification' else 5,
    scoring=scoring_metric,
    n_jobs=-1,
    random_state=42,
    verbose=0,
    refit=True
)
search.fit(X_train, y_train)

print(f"=== RandomizedSearchCV (30 itérations) ===")
print(f"Meilleur score ({scoring_metric}) : {search.best_score_:.4f}")
print(f"Meilleurs paramètres :")
for k, v in search.best_params_.items():
    print(f"  {k} : {v}")

best_pipe = search.best_estimator_
y_pred_best = best_pipe.predict(X_test)
if TASK == "classification":
    print(f"\nAUC-ROC test : {roc_auc_score(y_test, best_pipe.predict_proba(X_test)[:,1]):.4f}" if len(y.unique())==2 else "")
else:
    print(f"R² test : {r2_score(y_test, y_pred_best):.4f}")


In [ ]:
# ── Optuna — Bayesian Optimization ───────────────────────────────
OPTUNA_ENABLED = False   # !pip install optuna

if OPTUNA_ENABLED:
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        
        def objective(trial):
            n_est = trial.suggest_int('n_estimators', 100, 800)
            max_depth = trial.suggest_categorical('max_depth', [None, 5, 10, 20])
            min_leaf = trial.suggest_int('min_samples_leaf', 5, 50)
            
            clf = RandomForestClassifier(n_estimators=n_est, max_depth=max_depth,
                                          min_samples_leaf=min_leaf, n_jobs=-1, random_state=42)
            pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
            score = cross_val_score(pipe, X_train, y_train, cv=3,
                                    scoring=scoring_metric, n_jobs=-1).mean()
            return score
        
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=50)
        
        print(f"=== Optuna (50 trials) ===")
        print(f"Meilleur score : {study.best_value:.4f}")
        print(f"Meilleurs params : {study.best_params}")
        
    except ImportError:
        print("ℹ Optuna non installé : !pip install optuna")
else:
    print("ℹ Optuna désactivé. Activer OPTUNA_ENABLED = True.")


---
## 15. Cas d'Usage Complet — Bank Marketing Dataset

> Si vous n'utilisez pas le Bank Marketing Dataset, cette section sert de référence architecturale.


In [ ]:
# ── Feature Engineering Bank Marketing ───────────────────────────
BANK_DEMO = SOURCE == "bank_demo"

if BANK_DEMO:
    df_bank = df_raw.copy()
    
    # Feature engineering spécifique
    df_bank['was_contacted_before'] = (df_bank['pdays'] != -1).astype(int)
    df_bank['pdays'] = df_bank['pdays'].replace(-1, 0)
    
    month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
                 'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
    df_bank['month_num'] = df_bank['month'].map(month_map)
    df_bank['negative_balance'] = (df_bank['balance'] < 0).astype(int)
    df_bank['log_balance'] = np.log1p(df_bank['balance'].clip(lower=0))
    
    # Encoder la cible
    df_bank['y_bin'] = (df_bank['y'] == 'yes').astype(int)
    
    NUM_B = ['age','balance','duration','campaign','pdays','previous',
             'month_num','was_contacted_before','negative_balance','log_balance']
    CAT_B = ['job','marital','education','default','housing','loan','contact','poutcome']
    
    X_b = df_bank[NUM_B + CAT_B]
    y_b = df_bank['y_bin']
    
    X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
        X_b, y_b, test_size=0.2, stratify=y_b, random_state=42
    )
    
    neg_b, pos_b = (y_tr_b == 0).sum(), (y_tr_b == 1).sum()
    
    pre_b = build_preprocessor(NUM_B, CAT_B, scaler="standard")
    
    models_b = {
        'LR':  (Pipeline([('pre', pre_b),
                          ('clf', LogisticRegression(class_weight='balanced', C=1, max_iter=1000))]),),
        'RF':  (Pipeline([('pre', pre_b),
                          ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                                          max_depth=10, random_state=42, n_jobs=-1))]),),
    }
    
    cv_b = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    print("=== Bank Marketing — AUC-ROC par modèle ===")
    for name, (pipe,) in models_b.items():
        auc = cross_val_score(pipe, X_b, y_b, cv=cv_b, scoring='roc_auc', n_jobs=-1).mean()
        print(f"  {name:5s} : AUC = {auc:.4f}")
    
    print(
        "Resultats attendus (document) :
"
        "  Logistic Regression : AUC ~0.79
"
        "  Random Forest       : AUC ~0.83
"
        "  XGBoost             : AUC ~0.85-0.87 (apres tuning)

"
        "[!] 'duration' est la feature la plus predictive MAIS connue seulement
"
        "    apres l'appel => data leakage en production !
"
    )
else:
    print("ℹ Section Bank Marketing non active (SOURCE != 'bank_demo').")
    print("  Configurer SOURCE = 'bank_demo' pour charger le dataset de démonstration.")


---
## 16. Aide-Mémoire — Choisir le bon outil

### Validation croisée finale sur tous les modèles


In [ ]:
# ── Cross-validation finale — tous modèles ────────────────────────
print("=== VALIDATION CROISÉE FINALE ===")
print(f"Stratégie : {N_SPLITS}-fold | Scoring : {scoring_metric}")
print()

all_pipes = []
if TASK == "classification":
    cw = 'balanced' if IMBALANCE_STRATEGY == 'class_weight' else None
    all_pipes = [
        ("Logistic Regression", Pipeline([('pre', preprocessor),
            ('clf', LogisticRegression(class_weight=cw, C=1, max_iter=1000, random_state=42))])),
        ("Random Forest",       Pipeline([('pre', preprocessor),
            ('clf', RandomForestClassifier(n_estimators=200, class_weight=cw,
                                           n_jobs=-1, random_state=42))])),
        ("Decision Tree",       Pipeline([('pre', preprocessor),
            ('clf', DecisionTreeClassifier(max_depth=8, random_state=42))])),
        ("KNN",                 Pipeline([('pre', preprocessor),
            ('clf', KNeighborsClassifier(n_neighbors=5))])),
        ("Naive Bayes",         Pipeline([('pre', preprocessor), ('clf', GaussianNB())])),
    ]
    try:
        from xgboost import XGBClassifier
        neg = (y==0).sum(); pos = (y==1).sum()
        all_pipes.append(("XGBoost", Pipeline([('pre', preprocessor),
            ('clf', XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                                   scale_pos_weight=neg/pos if cw else 1,
                                   random_state=42, verbosity=0, n_jobs=-1))])))
    except: pass
else:
    all_pipes = [
        ("Linear Regression", Pipeline([('pre', preprocessor), ('reg', LinearRegression())])),
        ("Ridge",             Pipeline([('pre', preprocessor), ('reg', RidgeCV())])),
        ("Random Forest",     Pipeline([('pre', preprocessor),
            ('reg', RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42))])),
    ]

results_cv = {}
for name, pipe in all_pipes:
    cv_obj = StratifiedKFold(N_SPLITS, shuffle=True, random_state=42) if TASK == 'classification' else N_SPLITS
    scores = cross_val_score(pipe, X_train, y_train, cv=cv_obj,
                              scoring=scoring_metric, n_jobs=-1)
    results_cv[name] = scores
    print(f"  {name:25s} : {scores.mean():.4f} ± {scores.std():.4f}")

# Boxplot des scores CV
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([results_cv[n] for n in results_cv], labels=list(results_cv.keys()), vert=True)
ax.set_ylabel(scoring_metric)
ax.set_title(f"Cross-validation ({N_SPLITS}-fold) — Comparaison des modèles", fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Courbe d'apprentissage du meilleur modèle ────────────────────
best_name = max(results_cv, key=lambda n: results_cv[n].mean())
best_cv_pipe = dict(all_pipes)[best_name]
print(f"Meilleur modèle CV : {best_name}")

# Tracer la learning curve
from sklearn.model_selection import learning_curve
cv_lc = StratifiedKFold(5, shuffle=True, random_state=42) if TASK == 'classification' else 5

train_sizes, train_sc, val_sc = learning_curve(
    best_cv_pipe, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=cv_lc, scoring=scoring_metric, n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_sc.mean(1), 'o-', color='steelblue', label='Train')
ax.fill_between(train_sizes, train_sc.mean(1)-train_sc.std(1),
                train_sc.mean(1)+train_sc.std(1), alpha=0.2, color='steelblue')
ax.plot(train_sizes, val_sc.mean(1), 'o-', color='coral', label='Validation')
ax.fill_between(train_sizes, val_sc.mean(1)-val_sc.std(1),
                val_sc.mean(1)+val_sc.std(1), alpha=0.2, color='coral')
ax.set_xlabel("Taille du training set")
ax.set_ylabel(scoring_metric)
ax.set_title(f"Courbe d'apprentissage — {best_name}", fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 16.2 Aide-mémoire — Tableau récapitulatif

In [ ]:
print("""
AIDE-MÉMOIRE — CHOISIR L'ALGORITHME
=====================================
Logistic Regression  : Baseline, interprétabilité, données linéaires. Normaliser.
Random Forest        : Robustesse, peu de tuning, feature importance. Pas de normalisation.
XGBoost/LightGBM     : Meilleures performances tabulaires. Tuning nécessaire.
SVM                  : Haute dimension (texte), petits datasets. Obligatoirement normaliser.
MLP (Dense)          : Features complexes, beaucoup de données. Normaliser + Dropout.
CNN                  : Images, texte (1D-Conv). Transfer Learning si peu de données.
LSTM/GRU             : Séquences, séries temporelles. Lent sans GPU.
BERT/Transformers    : NLP SOTA. Lourd, GPU requis.
KMeans               : Clustering, exploration. Choisir k, sensible outliers.
Isolation Forest     : Détection anomalies. Régler contamination.

AIDE-MÉMOIRE — CHOISIR LA MÉTRIQUE
=====================================
Accuracy      : Classes équilibrées UNIQUEMENT
AUC-ROC       : Ranking probabilités, déséquilibre — métrique Kaggle par défaut
F1-score      : Compromis precision/recall
Precision     : Minimiser FP (spam, pub)
Recall        : Minimiser FN (maladie, fraude)
Log-Loss      : Calibration probabilités
RMSE          : Régression — sensible grosses erreurs
MAE           : Régression — robuste outliers

⚠ 6 ERREURS FRÉQUENTES À ÉVITER
===================================
1. Data leakage : StandardScaler avant split → TOUJOURS Pipeline
2. Mauvaise métrique : accuracy sur données déséquilibrées → AUC-ROC ou F1
3. Overfitting invisible : évaluer sur train seulement → cross-validation
4. Hyperparamètres sur test set → utiliser validation set ou CV
5. One-Hot sur ordinales → OrdinalEncoder avec ordre explicite
6. Oublier random_state → résultats non reproductibles (toujours 42)
""")


In [ ]:
# ── Sauvegarde du meilleur modèle ────────────────────────────────
import joblib

best_cv_pipe.fit(X_train, y_train)
joblib.dump(best_cv_pipe, 'best_model_ds52.pkl')
print(f"✓ Modèle '{best_name}' sauvegardé → best_model_ds52.pkl")

# Chargement
# loaded_model = joblib.load('best_model_ds52.pkl')
# predictions = loaded_model.predict(X_new)


---
## ✅ Pipeline DS52 complété

| Étape | Statut |
|-------|--------|
| 0. Processus & chargement | ✅ |
| 1. Types & encodage | ✅ |
| 2. EDA | ✅ |
| 3. Fondements statistiques | ✅ |
| 4. Preprocessing | ✅ |
| 5. Modèles classification | ✅ |
| 6. Métriques | ✅ |
| 7. Modèles régression | ✅ |
| 8. Deep Learning | ✅ (activer DL_ENABLED) |
| 9. NLP | ✅ (activer NLP_ENABLED) |
| 10. Big Data | ✅ (activer DASK/SPARK_ENABLED) |
| 11. XAI | ✅ (activer SHAP/LIME_ENABLED) |
| 12. Clustering & Anomalies | ✅ |
| 13. Séries temporelles | ✅ (activer TS_ENABLED) |
| 14. Hyperparamètres | ✅ |
| 15. Cas Bank Marketing | ✅ (SOURCE='bank_demo') |
| 16. Aide-mémoire | ✅ |

> **DS52 — UTBM 2026** — Ce notebook accompagne la fiche technique DS52.
